In [ ]:
# 0_download_and_prepare.py
import os
from datasets import load_dataset
import soundfile as sf
import argparse
from pathlib import Path
import tqdm

def main(out_dir="torgo_local", split_ratio=0.9):
    os.makedirs(out_dir, exist_ok=True)
    print("Loading TORGO dataset from HuggingFace mirror...")
    ds = load_dataset("abnerh/TORGO-database")  # if unavailable, replace with local download step
    # ds usually has 'train' split only; we'll create train/val split
    audios = ds["train"]
    items = []
    # Save each sample locally
    for i, sample in enumerate(tqdm.tqdm(audios)):
        audio = sample["audio"]["array"]
        sr = sample["audio"]["sampling_rate"]
        tr = sample.get("transcription", "") or sample.get("text", "") or ""
        speaker = sample.get("speaker", f"spk{i:04d}")
        fname = f"{speaker}_{i:06d}.wav"
        p = Path(out_dir) / fname
        sf.write(str(p), audio, sr)
        items.append({"path": str(p), "transcript": tr})
    # split
    split_idx = int(len(items) * split_ratio)
    train = items[:split_idx]
    val = items[split_idx:]
    import json
    with open(Path(out_dir)/"train.jsonl", "w", encoding="utf-8") as f:
        for it in train:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    with open(Path(out_dir)/"val.jsonl", "w", encoding="utf-8") as f:
        for it in val:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"Saved {len(train)} train and {len(val)} validation examples in {out_dir}")

if __name__ == "__main__":
    main()


hello world
